In [1]:
from torchvision import transforms
from PIL import Image
from pathlib import Path

from typing import Any, Callable, Dict, List, Optional, Tuple
import torch
from torchvision.datasets import VisionDataset
import numpy as np
import copy
import gc
import json
import logging
import os
import warnings
from torch.utils.data import DataLoader

from pathlib import Path

import dill
import flwr as fl
import numpy as np
import ray
import torch
from Utils.model_utils import ModelUtils
from Utils.train_parameters import TrainParameters
from Utils.utils import Utils
from flwr.common.typing import Scalar
from opacus import PrivacyEngine
from opacus.accountants.utils import get_noise_multiplier

from FairReg.Learning.learning import Learning
from FairReg.Regularization.RegularizationLoss import RegularizationLoss

/home/lcorbucci/Unfairness-Regularization/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-08-26 10:52:59,738	INFO util.py:159 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [4]:
Utils.rescale_lambda(
    value=0.1,
    old_min=0,
    old_max=1 - 0.5,
    new_min=0,
    new_max=1,
)

0.2

In [13]:
class TorchVision_FL(VisionDataset):
    """This is just a trimmed down version of torchvision.datasets.MNIST.

    Use this class by either passing a path to a torch file (.pt)
    containing (data, targets) or pass the data, targets directly
    instead.
    """

    def __init__(
        self,
        path_to_data=None,
        data=None,
        targets=None,
        transform: Optional[Callable] = None,
    ) -> None:
        path = path_to_data.parent if path_to_data else None
        self.dataset_path = path.parent.parent.parent if path_to_data else None

        super(TorchVision_FL, self).__init__(path, transform=transform)
        self.transform = transform

        if path_to_data:
            # load data and targets (path_to_data points to an specific .pt file)
            self.data, self.sensitive_features, self.targets = torch.load(path_to_data)
        else:
            self.data = data
            self.targets = targets

    def __getitem__(self, index: int) -> Tuple[Any, Any]:
        img, target = self.data[index], int(self.targets[index])

        # doing this so that it is consistent with all other datasets
        # to return a PIL Image
        if isinstance(img, str):
            path = self.dataset_path / "img_align_celeba/" / self.data[index]
            img = Image.open(path).convert(
                "RGB",
            )

        if not isinstance(img, Image.Image):  # if not PIL image
            if not isinstance(img, np.ndarray):  # if torch tensor
                img = img.numpy()

            img = Image.fromarray(img)

        if self.transform is not None:
            img = self.transform(img)

        if self.target_transform is not None:
            target = self.target_transform(target)

        sensitive_feature = self.sensitive_features[index]

        return img, sensitive_feature, target

    def __len__(self) -> int:
        return len(self.data)

In [14]:
path_to_data = Path("/home/lcorbucci/data/celeba/celeba-10-batches-py/federated_4/0/train.pt")

In [15]:
# TorchVision_FL(
#     path_to_data,
#     transform=transforms.Compose(
#         [
#             transforms.Resize((64, 64)),
#             transforms.ToTensor(),
#             transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
#         ],
#     ),
# )[0]

In [16]:
path_to_data = Path("../../../data/celeba/celeba-10-batches-py/federated/1/train.pt")

dataset = torch.load(path_to_data)

print(dataset[0])
kwargs = {"num_workers": 0, "pin_memory": True, "drop_last": False}
train_loader = DataLoader(dataset, batch_size=100, **kwargs)

delta = (1 / len(train_loader.dataset)) / 3
net = ModelUtils.get_model("celeba", device="cpu")

(
    private_net,
    private_optimizer,
    train_loader,
    privacy_engine,
) = Utils.create_private_model(
    model=net,
    epsilon=2.0,
    original_optimizer=torch.optim.Adam(
        net.parameters(),
        lr=0.1,
    ),
    train_loader=train_loader,
    epochs=10,
    delta=0.0001,
    MAX_GRAD_NORM=20,
    batch_size=100,
    noise_multiplier=None,
    accountant=None,
)

private_model_regularization = None
private_optimizer_regularization = None

['200145.jpg', '143649.jpg', '055933.jpg', '075815.jpg', '133630.jpg', '015103.jpg', '173744.jpg', '135785.jpg', '140550.jpg', '079725.jpg', '143978.jpg', '053403.jpg', '177325.jpg', '026378.jpg', '070486.jpg', '163302.jpg', '015640.jpg', '141531.jpg', '095931.jpg', '201039.jpg', '102855.jpg', '053291.jpg', '175988.jpg', '049088.jpg', '111786.jpg', '097392.jpg', '153955.jpg', '086078.jpg', '190534.jpg', '075780.jpg', '198409.jpg', '060471.jpg', '197760.jpg', '101798.jpg', '059295.jpg', '008153.jpg', '159740.jpg', '066232.jpg', '057898.jpg', '038497.jpg', '001533.jpg', '190312.jpg', '070320.jpg', '172859.jpg', '155744.jpg', '170438.jpg', '034855.jpg', '002769.jpg', '043229.jpg', '009465.jpg', '144811.jpg', '067595.jpg', '087683.jpg', '075137.jpg', '083651.jpg', '181150.jpg', '099523.jpg', '174105.jpg', '184431.jpg', '086172.jpg', '016999.jpg', '028410.jpg', '138000.jpg', '109299.jpg', '108959.jpg', '170852.jpg', '043690.jpg', '014725.jpg', '069322.jpg', '061786.jpg', '131627.jpg', '1834

In [17]:
train_loader = Utils.get_dataloader(
    "../../../data/celeba/celeba-10-batches-py/federated_4",
    "1",
    batch_size=100,
    workers=1,
    dataset="celeba",
    partition="train",
)

In [18]:
for item in train_loader:
    print(len(item))
    print(len(item[0]))
    print(item[0])
    break

3
100
tensor([[[[ 0.4745,  0.4745,  0.4745,  ...,  0.6157,  0.6078,  0.6157],
          [ 0.4745,  0.4745,  0.4745,  ...,  0.6157,  0.6078,  0.6078],
          [ 0.4745,  0.4745,  0.4745,  ...,  0.6157,  0.6157,  0.6078],
          ...,
          [-0.3098, -0.2471, -0.1373,  ...,  0.8118,  0.8118,  0.8118],
          [-0.4588, -0.4196, -0.3725,  ...,  0.8118,  0.8118,  0.8118],
          [-0.5137, -0.4588, -0.4667,  ...,  0.8118,  0.8118,  0.8118]],

         [[ 0.4824,  0.4824,  0.4824,  ...,  0.5843,  0.5765,  0.5843],
          [ 0.4824,  0.4824,  0.4824,  ...,  0.5843,  0.5765,  0.5765],
          [ 0.4824,  0.4824,  0.4824,  ...,  0.5843,  0.5843,  0.5765],
          ...,
          [-0.6235, -0.5294, -0.3961,  ...,  0.7804,  0.7804,  0.7804],
          [-0.7255, -0.6863, -0.6235,  ...,  0.7804,  0.7804,  0.7804],
          [-0.7333, -0.6863, -0.7176,  ...,  0.7804,  0.7804,  0.7804]],

         [[ 0.4431,  0.4431,  0.4431,  ...,  0.5608,  0.5529,  0.5608],
          [ 0.4431,  0.4

In [19]:
# path_to_data = Path("../../../data/dutch/federated/9/train.pt")

# dataset = torch.load(path_to_data)
# print(dataset[0])
# kwargs = {"num_workers": 0, "pin_memory": True, "drop_last": False}
# train_loader = DataLoader(dataset, batch_size=100, **kwargs)

# delta = (1 / len(train_loader.dataset)) / 3
# net = ModelUtils.get_model("celeba", device="cpu")

# (
#     private_net,
#     private_optimizer,
#     train_loader,
#     privacy_engine,
# ) = Utils.create_private_model(
#     model=net,
#     epsilon=2.0,
#     original_optimizer=torch.optim.Adam(
#         net.parameters(),
#         lr=0.1,
#     ),
#     train_loader=train_loader,
#     epochs=10,
#     delta=0.0001,
#     MAX_GRAD_NORM=20,
#     batch_size=100,
#     noise_multiplier=None,
#     accountant=None,
# )

# private_model_regularization = None
# private_optimizer_regularization = None

In [20]:
# for item in train_loader:
#     print(len(item))
#     print(len(item[0]))
#     print(item[0])
#     break